# Single-stage OMS experiment

One Stage, one Feature `x`. A part is machined to a Mean Setting `mu` with Normal Process Variation `sigma`.
After inspection it either **conforms** (L ≤ x ≤ U), is **reworked** (on the Rework Side of the limits; it goes
round the Stage again) or is **scrapped** (on the other side).

1. Set the inputs in the cell below. Every other cell reads them.
2. **Run All**. The notebook optimises `mu` and plots how profit, cost and outcomes change as `mu` moves.

The defaults are thesis reference case R2 (Table 3.3), so the optimiser should return mu ≈ 10.61 for profit
and mu ≈ 10.09 for production cost.

In [ ]:
# ---- Inputs: change these, then Run All ----
LOWER, UPPER = 8.0, 12.0     # Specification Limits of x
SIGMA = 1.0                  # Process Variation (standard deviation)
REWORK_SIDE = "upper"        # "upper": above U is reworked, below L scrapped; "lower": the reverse; "none": both scrapped

SELLING_PRICE = 120.0        # SP, paid for each conforming part
PROCESSING_COST = 25.0       # PC, per part entering the Stage
REWORK_COST = 10.0           # RC, per rework pass
SCRAP_COST = 25.0            # SC, per scrapped part (same whether scrapped first time or after rework)

MU_SWEEP = (LOWER, UPPER)    # range of Mean Settings to plot (rework cost explodes far past the Rework Side)
SIGMA_SWEEP = (0.25, 1.75)   # range of Process Variation for the sensitivity study

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from oms import CostTable, Feature, MeanSettings, MultivariateNormal, ReworkSide, Sequence, Stage
from oms import evaluate_sequence, optimise_means

# Palette slots 1-3, safe for colour-blind readers in any pairing. Each colour means one outcome throughout:
# blue = rework, orange = scrap, aqua = conform; grey = processing (the fixed baseline).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED, NEUTRAL = "#1f2328", "#6e7781", "#afb8c1"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": "#d0d7de", "grid.linewidth": 0.6, "lines.linewidth": 2,
    "legend.frameon": False,
})

## Model

In [ ]:
INITIAL, REWORK = frozenset(), frozenset({"x"})


def build_sequence(sigma=SIGMA):
    side = ReworkSide(REWORK_SIDE)
    rework = {REWORK: REWORK_COST} if side is not ReworkSide.NONE else {}
    scrap = {INITIAL: SCRAP_COST} | ({REWORK: SCRAP_COST} if side is not ReworkSide.NONE else {})
    stage = Stage(
        features=(Feature("x", lower=LOWER, upper=UPPER, rework_side=side),),
        variation=MultivariateNormal([sigma]),
        costs=CostTable(processing=PROCESSING_COST, rework=rework, scrap=scrap),
    )
    return Sequence(stages=(stage,), selling_price=SELLING_PRICE)


sequence = build_sequence()
stage = sequence.stages[0]

print(f"{'Specification Limits':<24}[{LOWER}, {UPPER}]")
print(f"{'Process Variation':<24}sigma = {SIGMA}")
print(f"{'Rework Side':<24}{REWORK_SIDE}")
print(f"{'Selling price':<24}{SELLING_PRICE}")
print(f"{'Processing cost':<24}{PROCESSING_COST}")
print(f"{'Rework cost':<24}{REWORK_COST}")
print(f"{'Scrap cost':<24}{SCRAP_COST}")
print(f"{'States':<24}{[sorted(s) or 'Initial' for s in stage.states]}")

In [ ]:
print("hello")

## Optimise the Mean Setting

In [ ]:
best_profit = optimise_means(sequence)
best_cost = optimise_means(sequence, objective="production_cost")
MU_PROFIT = best_profit.means[0].by_state[INITIAL]["x"]
MU_COST = best_cost.means[0].by_state[INITIAL]["x"]


def show(title, result, mu):
    b = result.breakdown
    print(title)
    print(f"  Mean Setting       {mu:9.4f}")
    print(f"  Expected Profit    {b.expected_profit:9.4f}")
    print(f"  Final Conformance  {b.final_conformance:9.4f}")
    print(f"  Production cost    {b.production_cost:9.4f}  = processing {b.processing_costs[0]:.4f}"
          f" + scrap {b.scrap_costs[0]:.4f} + rework {b.rework_costs[0]:.4f}")
    print(f"  [{result.method}, {result.n_starts} starts, success={result.success}]\n")


show("Maximum Expected Profit", best_profit, MU_PROFIT)
show("Minimum production cost", best_cost, MU_COST)

## Process distribution at the profit-optimal mean

The first-pass distribution of `x`. The shaded areas are the probabilities of each inspection outcome.

In [ ]:
def regions(side):
    # (label, colour) for below L and above U
    scrap, rework = ("Scrap", ORANGE), ("Rework", BLUE)
    return {"upper": (scrap, rework), "lower": (rework, scrap), "none": (scrap, scrap)}[side]


def plot_distribution(mu, ax):
    xs = np.linspace(min(LOWER, mu) - 4 * SIGMA, max(UPPER, mu) + 4 * SIGMA, 600)
    pdf = norm.pdf(xs, mu, SIGMA)
    (below_label, below_c), (above_label, above_c) = regions(REWORK_SIDE)
    p_below, p_above = norm.cdf(LOWER, mu, SIGMA), norm.sf(UPPER, mu, SIGMA)
    ax.fill_between(xs, pdf, where=xs <= LOWER, color=below_c, alpha=0.35, lw=0,
                    label=f"{below_label} (below L)  {p_below:.3f}")
    ax.fill_between(xs, pdf, where=(xs >= LOWER) & (xs <= UPPER), color=AQUA, alpha=0.35, lw=0,
                    label=f"Conform  {1 - p_below - p_above:.3f}")
    ax.fill_between(xs, pdf, where=xs >= UPPER, color=above_c, alpha=0.35, lw=0,
                    label=f"{above_label} (above U)  {p_above:.3f}")
    ax.plot(xs, pdf, color=INK, lw=1.5)
    for limit, name in ((LOWER, "L"), (UPPER, "U")):
        ax.axvline(limit, color=MUTED, lw=1, ls="--")
        ax.annotate(name, (limit, pdf.max() * 1.05), ha="center", color=MUTED)
    ax.axvline(mu, color=INK, lw=1)
    ax.annotate(f"mu = {mu:.3f}", (mu, pdf.max() * 1.12), ha="center", color=INK)
    ax.set_ylim(0, pdf.max() * 1.2)
    ax.set_xlabel("x")
    ax.set_ylabel("density")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3)


fig, ax = plt.subplots(figsize=(8, 3.6))
plot_distribution(MU_PROFIT, ax)
ax.set_title("First-pass distribution at the profit-optimal Mean Setting", loc="left")
plt.show()

## Sweep the Mean Setting

Evaluate the Stage at every `mu` in `MU_SWEEP` (no optimisation, just the model) to see the shape of the
trade-off around the optimum.

In [ ]:
mus = np.linspace(*MU_SWEEP, 281)
sweep = [evaluate_sequence(sequence, [MeanSettings.per_feature(stage, {"x": float(m)})]) for m in mus]

profit = np.array([b.expected_profit for b in sweep])
cost = np.array([b.production_cost for b in sweep])
processing = np.array([b.processing_costs[0] for b in sweep])
scrap = np.array([b.scrap_costs[0] for b in sweep])
rework = np.array([b.rework_costs[0] for b in sweep])
conformance = np.array([b.final_conformance for b in sweep])
scrap_prob = np.array([b.stages[0].scrap_probability for b in sweep])
rework_visits = np.array([b.stages[0].visits.get(REWORK, 0.0) for b in sweep])


def mark_limits(ax):
    for limit in (LOWER, UPPER):
        ax.axvline(limit, color=MUTED, lw=1, ls="--")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)
ax1.plot(mus, profit, color=BLUE)
ax1.plot(MU_PROFIT, best_profit.breakdown.expected_profit, "o", ms=8, color=BLUE, mec="white", mew=2)
ax1.annotate(f"max {best_profit.breakdown.expected_profit:.2f}\nat mu = {MU_PROFIT:.3f}",
             (MU_PROFIT, best_profit.breakdown.expected_profit), xytext=(10, -30), textcoords="offset points")
ax1.set_title("Expected Profit per part", loc="left")

ax2.plot(mus, cost, color=ORANGE)
ax2.plot(MU_COST, best_cost.breakdown.production_cost, "o", ms=8, color=ORANGE, mec="white", mew=2)
ax2.annotate(f"min {best_cost.breakdown.production_cost:.2f}\nat mu = {MU_COST:.3f}",
             (MU_COST, best_cost.breakdown.production_cost), xytext=(10, 20), textcoords="offset points")
ax2.set_title("Production cost per part (processing + scrap + rework)", loc="left")
for ax in (ax1, ax2):
    mark_limits(ax)
    ax.set_xlabel("Mean Setting mu")
plt.tight_layout()
plt.show()

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)

ax1.stackplot(mus, processing, scrap, rework, colors=[NEUTRAL, ORANGE, BLUE], alpha=0.85,
              labels=["Processing", "Scrap", "Rework"], edgecolor="white", linewidth=1)
ax1.set_title("Where the production cost comes from", loc="left")
ax1.set_ylabel("cost per part")
ax1.legend(loc="upper center", ncol=3)

ax2.plot(mus, conformance, color=AQUA, label="Final Conformance")
ax2.plot(mus, scrap_prob, color=ORANGE, label="Scrap probability")
ax2.set_ylim(0, 1.02)
ax2.set_title("Outcome probabilities per part", loc="left")
ax2.legend(loc="center left")

ax3.plot(mus, rework_visits, color=BLUE)
ax3.set_title("Expected rework passes per part", loc="left")

for ax in (ax1, ax2, ax3):
    mark_limits(ax)
    ax.axvline(MU_PROFIT, color=INK, lw=1, alpha=0.5)
    ax.set_xlabel("Mean Setting mu")
plt.tight_layout()
plt.show()

## Sensitivity: how does Process Variation move the optimum?

Re-optimise for each `sigma` in `SIGMA_SWEEP`. A tighter process lets the mean sit closer to the middle;
a wider one pushes it towards the Rework Side, where an out-of-spec part costs less than scrap.

At small `sigma` almost every part conforms wherever `mu` sits inside the limits, so the profit curve is flat
and the optimiser can return any point on the plateau: a wobble in `mu*` there is not meaningful.

In [ ]:
sigmas = np.linspace(*SIGMA_SWEEP, 13)
runs = [optimise_means(build_sequence(s)) for s in sigmas]
opt_mu = np.array([r.means[0].by_state[INITIAL]["x"] for r in runs])
opt_profit = np.array([r.breakdown.expected_profit for r in runs])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True)
ax1.plot(sigmas, opt_mu, "o-", ms=6, color=BLUE, mec="white", mew=1.5)
ax1.axhline(LOWER, color=MUTED, lw=1, ls="--")
ax1.axhline(UPPER, color=MUTED, lw=1, ls="--")
ax1.axhline((LOWER + UPPER) / 2, color=MUTED, lw=1, ls=":")
ax1.set_title("Profit-optimal Mean Setting", loc="left")
ax1.set_ylabel("mu")
ax2.plot(sigmas, opt_profit, "o-", ms=6, color=BLUE, mec="white", mew=1.5)
ax2.set_title("Maximum Expected Profit per part", loc="left")
for ax in (ax1, ax2):
    ax.axvline(SIGMA, color=INK, lw=1, alpha=0.5)
    ax.set_xlabel("Process Variation sigma")
plt.tight_layout()
plt.show()

for s, m, p in zip(sigmas, opt_mu, opt_profit):
    print(f"sigma {s:5.3f}   mu* {m:8.4f}   E(PR) {p:9.4f}")